# 01. Exploratory Data Analysis (EDA)
## AI Resume–Job Description Semantic Matching System

This notebook inspects the raw dataset `michaelozon/candidate-matching-synthetic`, explores role and seniority distributions, analyzes skill occurrences, and computes text length distributions.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))
from src.data_loader import load_raw_dataset, inspect_dataset_schema
from src.preprocessing import calculate_text_statistics

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')


### 1. Load Raw Resumes Dataset


In [ ]:
df = load_raw_dataset()
print('Dataframe shape:', df.shape)
df.head()


### 2. Dataset Schema & Missing Values Analysis


In [ ]:
schema = inspect_dataset_schema(df)
print('Schema Summary:')
for k, v in schema.items():
    print(f'{k}: {v}')

print('\nMissing values check:')
print(df.isnull().sum())


### 3. Role, Seniority, and Industry Distributions


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, y='role', order=df['role'].value_counts().index[:12], ax=axes[0], palette='viridis')
axes[0].set_title('Top 12 Candidate Roles', fontsize=12, fontweight='bold')

sns.countplot(data=df, x='seniority', order=['Junior', 'Mid', 'Senior'], ax=axes[1], palette='Set2')
axes[1].set_title('Seniority Distribution', fontsize=12, fontweight='bold')

sns.countplot(data=df, y='industry', order=df['industry'].value_counts().index, ax=axes[2], palette='mako')
axes[2].set_title('Industry Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


### 4. Technical Skill Frequency Analysis


In [ ]:
all_skills = [s for sk_list in df['skills'] for s in (sk_list if isinstance(sk_list, list) else [])]
skill_counts = Counter(all_skills)

top_skills_df = pd.DataFrame(skill_counts.most_common(25), columns=['Skill', 'Count'])

plt.figure(figsize=(12, 6))
sns.barplot(data=top_skills_df, x='Count', y='Skill', palette='rocket')
plt.title('Top 25 Most Frequent Skills Across Candidate Resumes', fontsize=14, fontweight='bold')
plt.xlabel('Frequency Count', fontsize=12)
plt.tight_layout()
plt.show()


### 5. Resume Text Length Statistics


In [ ]:
from src.preprocessing import format_resume_text

df['formatted_resume'] = df.apply(format_resume_text, axis=1)
df['char_count'] = df['formatted_resume'].apply(len)
df['word_count'] = df['formatted_resume'].apply(lambda x: len(x.split()))

print('Resume Word Count Summary:')
print(df['word_count'].describe())

plt.figure(figsize=(10, 4))
sns.histplot(df['word_count'], bins=30, kde=True, color='royalblue')
plt.title('Distribution of Resume Word Counts', fontsize=14, fontweight='bold')
plt.xlabel('Word Count', fontsize=12)
plt.tight_layout()
plt.show()
